In [1]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()

import altair as alt
import pandas as pd
import requests

In [2]:
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")

In [3]:
# Define cities
cities = {
    'Paris': (46, 2),
    'London': (51.5085, -0.1257),
    'Berlin': (52.5244, 13.4105),
    'Warsaw': (52.2298, 21.0118),
    'Prague': (50.088, 14.4208)
}

dfs = []

# Download max temperature values across cities from Open-Meteo
for city, (lat, lon) in cities.items():
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "1980-01-01",
        "end_date": "2026-06-28",
        "daily": "temperature_2m_max",
        "timezone": "auto"
    }
    response = requests.get(url, params=params)
    df = pd.DataFrame(response.json()['daily'])
    df['date'] = pd.to_datetime(df['time'])
    df['year'] = df['date'].dt.year
    df['city'] = city
    dfs.append(df)

In [4]:
# Combine city dataframes
temp_df = pd.concat(dfs, ignore_index=True)

# Create df for days above 35°C per city per year
heatwave_df = temp_df[temp_df['temperature_2m_max'] >= 35].groupby(['year', 'city']).size().reset_index(name='hot_days')

In [5]:
# Create complete grid of all years and cities
all_years = range(1980, 2027)
all_cities = heatwave_df['city'].unique()

# Re-index heatwave df to show 0 for years with no days above 35°C
full_index = pd.MultiIndex.from_product([all_years, all_cities], names=['year', 'city'])
heatwave_df = heatwave_df.set_index(['year', 'city']).reindex(full_index, fill_value=0).reset_index()

# Create rolling average to show underlying trend
heatwave_df = heatwave_df.sort_values(['city', 'year'])
heatwave_df['hot_days_rolling'] = heatwave_df.groupby('city')['hot_days'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())

In [6]:
# Create line chart for number of days above 35°C across cities
# Add faint line for raw days
raw_line = alt.Chart(heatwave_df).mark_line(opacity=0.2).encode(
    x=alt.X(
        'year:Q',
        title=None,
        scale=alt.Scale(domain=[1980,2027]),
        axis=alt.Axis(format='d', grid=False)),
    y=alt.Y(
        'hot_days:Q',
        title='Days above 35°C',
        axis=alt.Axis(grid=False)),
    color=alt.Color('city:N', legend=None)
)
# Add smooth line for rolling 5 year average
smooth_line = alt.Chart(heatwave_df).mark_line().encode(
    x=alt.X(
        'year:Q',
        title=None,
        scale=alt.Scale(domain=[1980,2026]),
        axis=alt.Axis(format='d', grid=False)),
    y=alt.Y('hot_days_rolling:Q'),
    color=alt.Color('city:N', legend=None)
)
# Add city labels
last_points = heatwave_df[heatwave_df['year'] == heatwave_df['year'].max()].copy()
label_offsets = {
    'Paris': 0,
    'London': 0.1,
    'Berlin': 0,
    'Warsaw': -0.1,
    'Prague': 0
}
last_points['y_offset'] = last_points['hot_days_rolling'] + last_points['city'].map(label_offsets)
labels = alt.Chart(last_points).mark_text(
    align='left',
    dx=5,
    fontSize=11
).encode(
    x=alt.X('year:Q', scale=alt.Scale(domain=[1980,2026])),
    y=alt.Y('y_offset:Q'),
    text=alt.Text('city:N'),
    color=alt.Color('city:N', legend=None)
)

# Add source
smooth_line = alt.layer(smooth_line).properties(
    title=alt.TitleParams(
        ['Source: Open-Meteo','Light lines show annual counts; solid lines show the 10-year rolling average.'],
        orient='bottom',
        fontStyle='italic',
        fontSize=10,
        color='#676A8680',
        fontWeight='normal',
        frame='group',
        offset=7
    )
)
# Add title
title = alt.Chart({'values': [{}]}).mark_text(
    text="European cities that rarely saw extreme heat now face a new norm",
    align='left',
    baseline='top',
    fontSize=15
).encode(
    x=alt.value(-25),
    y=alt.value(-35)
)

heatwave_line = raw_line + smooth_line + labels + title
heatwave_line

alt.LayerChart(...)

Across Europe, extreme heat days that were once rare exceptions are becoming increasingly common. Warsaw recorded its first ever day above 35°C only last year. London had none until 2019. Paris saw 5 such days in 2026 alone, up from zero before 2003.

In [7]:
# Save charts
styles.save(heatwave_line, 'visualisation', 'heatwave_line_chart', width=450, height=360)